<a href="https://colab.research.google.com/github/duttaprat/BMI_503/blob/main/class_2/GENCODE_notebook.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Genomic Annotations GENCODE

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/duttaprat/BMI_503/blob/main/class_1/notebook1_genomics_sequence_analysis.ipynb)

**Course**: BMI 503 - Introduction to Computer Science for Biomedical Informatics  
**Instructors**: Pratik Dutta   
**Institution**: Stony Brook University

---



## Setup & Installation

In [ ]:
# Install required packages
!pip install pandas biopython requests -q

import pandas as pd
import re
from collections import defaultdict
import matplotlib.pyplot as plt
import seaborn as sns

print("✅ Setup complete!")

##  Downloading and Parsing GENCODE Data

### Where to Get GENCODE Data?

**GENCODE Website:** https://www.gencodegenes.org/

In [ ]:
import urllib.request
import gzip


# Full genome GTF is ~1.5 GB!
# URL for human GRCh38 release 49
url = "https://ftp.ebi.ac.uk/pub/databases/gencode/Gencode_human/release_49/gencode.v49.annotation.gtf.gz"

print("📥 Downloading GENCODE GTF...")
output_file = "gencode.v49.annotation.gtf.gz"
urllib.request.urlretrieve(url, output_file)
print(f"✅ Downloaded: {output_file}")

In [ ]:
# Parse only chromosome 22
print("\n📖 Parsing GTF file (chr22 only)...")

data = []
target_chr = 'chr22'  # Change to 'chr1', 'chr17', etc.

with gzip.open(output_file, 'rt') as f:
    for line in f:
        if line.startswith('#'):
            continue

        fields = line.strip().split('\t')
        if len(fields) < 9:
            continue

        # Only process target chromosome
        if fields[0] != target_chr:
            continue

        record = {
            'seqname': fields[0],
            'source': fields[1],
            'feature': fields[2],
            'start': int(fields[3]),
            'end': int(fields[4]),
            'score': fields[5],
            'strand': fields[6],
            'frame': fields[7],
        }

        # Parse attributes
        attributes = fields[8]
        for match in [
            ('gene_id', r'gene_id "([^"]+)"'),
            ('gene_name', r'gene_name "([^"]+)"'),
            ('transcript_id', r'transcript_id "([^"]+)"'),
            ('gene_type', r'gene_type "([^"]+)"')
        ]:
            m = re.search(match[1], attributes)
            if m:
                record[match[0]] = m.group(1)

        data.append(record)

df = pd.DataFrame(data)
print("✅ GTF parsed into DataFrame!")
print(f"\nShape: {df.shape[0]} rows × {df.shape[1]} columns")
print(f"\nColumns: {list(df.columns)}")
df

In [ ]:
print(f"✅ Parsed {target_chr}!")
print(f"   Features: {len(df):,}")
print(f"   Genes: {df[df['feature'] == 'gene'].shape[0]:,}")

In [ ]:
# Basic sequence operations
dna = Seq("ATGGCCATTGTAATGGGCCGCTGAAAGGGTGCCCGATAG")
print(f"DNA: {dna}")
print(f"Complement: {dna.complement()}")
print(f"Reverse Complement: {dna.reverse_complement()}")
print(f"RNA: {dna.transcribe()}")
print(f"Protein: {dna.transcribe().translate()}")
print(f"GC Content: {SeqUtils.gc_fraction(dna)*100:.2f}%")




### Step-by-Step Breakdown:

**Original DNA:**
```
ATGGCCATTGTAATGGGCCGCTGAAAGGGTGCCCGATAG
```

**Step 1: `dna.transcribe()`** - DNA → RNA
```
AUGGCCAUUGUAAUGGGCCGCUGAAAGGGUGCCCGAUAG
```
- T → U (Thymine becomes Uracil)
- Everything else stays the same

**Step 2: `.translate()`** - RNA → Protein

The genetic code reads RNA in **triplets (codons)**:

| Codon | Amino Acid | Letter Code |
|-------|------------|-------------|
| **AUG** | Methionine | **M** (Start codon) |
| **GCC** | Alanine | **A** |
| **AUU** | Isoleucine | **I** |
| **GUA** | Valine | **V** |
| **AUG** | Methionine | **M** |
| **GGC** | Glycine | **G** |
| **CGC** | Arginine | **R** |
| **UGA** | **STOP** | **\*** |
| AAG | Lysine | K |
| GGU | Glycine | G |
| GCC | Alanine | A |
| CGA | Arginine | R |
| UAG | **STOP** | **\*** |

**Result:**
```
M  A  I  V  M  G  R  *  K  G  A  R  *
```

**Full output:**
```
Protein: MAIVMGR*KGAR*
```

In [ ]:
# Download from NCBI
Entrez.email = "your.email@example.com"
handle = Entrez.efetch(db="nucleotide", id="NM_000546", rettype="fasta", retmode="text")
record = SeqIO.read(handle, "fasta")
handle.close()
print(f"Downloaded: {record.description[:6000]}")
print(f"Length: {len(record.seq)} bp")
print(f"First 100 bp: " + str(record.seq[:100]))

## Part 2: Working with Alignment Files - SAM and BAM

### What are SAM and BAM files?

When DNA sequencing is performed, millions of short DNA sequences (called **reads**) are generated. These reads need to be **aligned** or **mapped** to a reference genome to determine where each read came from in the genome.

**SAM** (Sequence Alignment/Map) and **BAM** (Binary Alignment/Map) files store this alignment information.

---

### SAM vs BAM

| Feature | SAM | BAM |
|---------|-----|-----|
| **Format** | Text (human-readable) | Binary (compressed) |
| **File Size** | Large (1-10 GB) | Small (100 MB - 1 GB) |
| **Speed** | Slow to process | Fast to process |
| **Use Case** | Viewing, debugging | Storage, analysis |
| **Extension** | `.sam` | `.bam` |

**Key Point**: BAM is just a compressed binary version of SAM. They contain the same information!

---

### What Information Do They Store?

Each alignment (read) contains:

1. **Read ID**: Unique identifier for the sequenced fragment
2. **Chromosome/Reference**: Which chromosome the read mapped to (e.g., chr1, chr2)
3. **Position**: Exact genomic position where the read aligns (e.g., position 12345)
4. **Mapping Quality**: How confident we are about this alignment (0-60 score)
5. **CIGAR String**: Describes the alignment (matches, insertions, deletions)
6. **Sequence**: The actual DNA sequence of the read
7. **Quality Scores**: Quality of each base call
8. **Flags**: Information about pairing, strand, etc.

---

### Example: What Does a SAM File Look Like?
```
@HD	VN:1.0	SO:coordinate
@SQ	SN:chr1	LN:249250621
read1	0	chr1	12345	60	50M	*	0	0	ATCGATCG...	IIIIIIII...
read2	16	chr1	12350	60	50M	*	0	0	GCTAGCTA...	IIIIIIII...
```

**Line-by-line explanation:**
- **@HD**: Header line (version info)
- **@SQ**: Reference sequence info (chromosome name and length)
- **read1, read2**: Actual alignment records

---

### Why Do We Need SAM/BAM Files?

SAM/BAM files are essential for:

1. **Variant Calling**: Finding SNPs, insertions, deletions
2. **RNA-seq Analysis**: Measuring gene expression levels
3. **ChIP-seq**: Identifying protein-DNA binding sites
4. **Coverage Analysis**: How well each genomic region is sequenced
5. **Quality Control**: Assessing sequencing and alignment quality

---

### Common Workflow
```
FASTQ files (raw reads)
        ↓
    Alignment Tool (BWA, Bowtie2)
        ↓
    SAM file (text output)
        ↓
    Convert to BAM (samtools)
        ↓
    BAM file (compressed, indexed)
        ↓
    Analysis (variant calling, coverage, etc.)
```

---

### BAM File Index (.bai)

BAM files are often paired with an **index file** (`.bam.bai`) that allows:
- **Fast random access** to specific genomic regions
- **Quick retrieval** of reads without scanning the entire file
- **Efficient visualization** in genome browsers

**Without index**: Must scan entire file sequentially (slow!)  
**With index**: Jump directly to region of interest (fast!)

---

### Tools for Working with SAM/BAM

| Tool | Purpose |
|------|---------|
| **samtools** | Swiss-army knife for SAM/BAM manipulation |
| **pysam** | Python interface to SAM/BAM files |
| **IGV** | Visualization tool (Integrative Genomics Viewer) |
| **Picard** | Java tools for SAM/BAM processing |

---

### What We'll Learn with Pysam

Using the `pysam` Python library, you'll learn to:

1. ✅ Open and read BAM files
2. ✅ Extract alignment information (chromosome, position, quality)
3. ✅ Calculate coverage (how many reads cover each position)
4. ✅ Filter reads based on quality or mapping
5. ✅ Count total mapped reads

Let's dive into the code! 👇

In [ ]:
# Small BAM file from 1000 Genomes
!wget ftp://ftp.1000genomes.ebi.ac.uk/vol1/ftp/phase3/data/HG00096/alignment/HG00096.chrom20.ILLUMINA.bwa.GBR.low_coverage.20120522.bam -O sample.bam


In [ ]:
pysam.index("sample.bam")

# Now you can use it
bam = pysam.AlignmentFile("sample.bam", "rb")
print(f"References: {bam.references}")
print(f"Total reads: {bam.count()}")
bam.close()

---
## Part 3: PyVCF (Variants)

In [ ]:
# Download VCF
!wget https://raw.githubusercontent.com/jamescasbon/PyVCF/master/vcf/test/example-4.0.vcf -O sample.vcf

vcf_reader = vcf.Reader(open('sample.vcf', 'r'))
print(f"Samples: {vcf_reader.samples}")

for i, record in enumerate(vcf_reader):
    if i >= 3: break
    print(f"{record.CHROM}:{record.POS} {record.REF}>{record.ALT[0]} Q={record.QUAL}")

In [ ]:
df= pd.read_csv("sample.vcf", sep='\t', comment='#', header=None)
# You can manually assign column names (standard VCF fields)
df.columns = [
    "CHROM", "POS", "ID", "REF", "ALT",
    "QUAL", "FILTER", "INFO", "FORMAT",
    "Sample1", "Sample2", "Sample3"
]

df

---
## Part 4: Pybedtools

In [ ]:
# Create BED files
with open('features.bed', 'w') as f:
    f.write("chr1\t100\t200\tfeature1\n")
    f.write("chr1\t300\t400\tfeature2\n")
    f.write("chr1\t500\t600\tfeature2\n")

with open('genes.bed', 'w') as f:
    f.write("chr1\t150\t250\tgene1\n")
    f.write("chr1\t350\t450\tgene1\n")

features = pybedtools.BedTool('features.bed')
genes = pybedtools.BedTool('genes.bed')
intersect = features.intersect(genes)
print("Intersections:")
print(intersect)

---
## Part 5: Scikit-bio

In [ ]:
# Sequence analysis
seq = DNA('ATGGCCATTGTAATGGGCCGCTGAAAGGGTGCCCGATAG')
print(f"Sequnce len: {len(seq)}")
print(f"GC content: {seq.gc_content():.2%}")
print(f"Reverse complement: {seq.reverse_complement()}")

# k-mers
kmers = list(seq.iter_kmers(3))
print(f"\nTotal 3-mers: {len(kmers)}")
kmer_counts = Counter(str(k) for k in kmers)
print(f"Most common: {kmer_counts.most_common(3)}")

kmers = list(seq.iter_kmers(6))
print(f"\nTotal 6-mers: {len(kmers)}")
kmer_counts = Counter(str(k) for k in kmers)
print(f"Most common: {kmer_counts.most_common(6)}")

---
## Part 6: Pandas

In [ ]:
# Genomic dataframe
data = {
    'Gene': ['GENE1', 'GENE2', 'GENE3'],
    'Chr': ['chr1', 'chr1', 'chr2'],
    'Start': [1000, 5000, 2000],
    'End': [2000, 6500, 3500],
    'Expression': [45.2, 123.5, 67.8]
}
df = pd.DataFrame(data)
df['Length'] = df['End'] - df['Start']
print(df)
print(f"\nMean expression: {df['Expression'].mean():.2f}")

---
## Part 7: Visualization

In [ ]:
# Nucleotide composition
seq = Seq("ATGGCCATTGTAATGGGCCGCTGAAAGGGTGCCCGATAG" * 3)
counts = [seq.count(n) for n in ['A', 'T', 'G', 'C']]

plt.figure(figsize=(10, 5))
plt.bar(['A', 'T', 'G', 'C'], counts, color=['#FF6B6B', '#4ECDC4', '#FFD93D', '#95E1D3'])
plt.title('Nucleotide Composition', fontweight='bold')
plt.ylabel('Count')
plt.grid(alpha=0.3)
plt.show()

In [ ]:
# GC content sliding window
sequence = Seq("ATGGCCATTGTAATGGGCCGCTGAAAGGGTGCCCGATAG" * 10)
window = 20
positions, gc_vals = [], []

for i in range(0, len(sequence)-window, 5):
    win = sequence[i:i+window]
    positions.append(i)
    gc_vals.append(SeqUtils.gc_fraction(win) * 100)

plt.figure(figsize=(12, 5))
plt.plot(positions, gc_vals, linewidth=2, color='#2E86AB')
plt.axhline(50, color='red', linestyle='--', alpha=0.5)
plt.title('GC Content Sliding Window', fontweight='bold')
plt.xlabel('Position')
plt.ylabel('GC %')
plt.grid(alpha=0.3)
plt.show()

---
## Complete Workflow Example

In [ ]:
print("="*60)
print("COMPLETE GENOMIC ANALYSIS WORKFLOW")
print("="*60)

# 1. Download sequence
print("\n[1] Downloading TP53 from NCBI...")
Entrez.email = "your.email@example.com"
handle = Entrez.efetch(db="nucleotide", id="NM_000546", rettype="fasta", retmode="text")
record = SeqIO.read(handle, "fasta")
handle.close()
print(f"✅ {record.description[:50]}... ({len(record.seq)} bp)")

# 2. Analyze
print("\n[2] Analyzing sequence properties...")
gc = SeqUtils.gc_fraction(record.seq) * 100
print(f"✅ GC Content: {gc:.2f}%")

# 3. Find motifs
print("\n[3] Finding TATA boxes...")
tata_pos = [i for i in range(len(record.seq)-6) if record.seq[i:i+6] == "TATAAA"]
print(f"✅ Found {len(tata_pos)} TATA box(es)")

# 4. k-mers
print("\n[4] Generating k-mers...")
dna_skbio = DNA(str(record.seq[:100]))
kmers = list(dna_skbio.iter_kmers(3))
print(f"✅ Generated {len(kmers)} 3-mers")

# 5. Create dataframe
print("\n[5] Creating summary dataframe...")
summary = pd.DataFrame({
    'Gene': ['TP53'],
    'Length': [len(record.seq)],
    'GC%': [gc],
    'TATA_boxes': [len(tata_pos)]
})
print(summary)

print("\n✅ Workflow complete!")
print("="*60)

---
## Exercises

### Exercise 1: Sequence Analysis
Download BRCA1 (NM_007294) and calculate GC content, find ATG positions

In [ ]:
# Your code here


### Exercise 2: VCF Analysis
Filter variants with quality > 50 and create a pandas DataFrame

In [ ]:
# Your code here


### Exercise 3: k-mer Analysis  
Generate 4-mers and find the most frequent k-mer

In [ ]:
# Your code here


---
## Summary


- ✅ Biopython for sequences
- ✅ Pysam for BAM files
- ✅ PyVCF for variants
- ✅ Pybedtools for intervals
- ✅ Scikit-bio for analysis
- ✅ Pandas for data
- ✅ Matplotlib/Seaborn for viz

### Resources
- [Biopython](http://biopython.org)
- [Pysam Docs](https://pysam.readthedocs.io)
- [NCBI](https://www.ncbi.nlm.nih.gov)